In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_df = spark.table(
    "customer360_dev.bronze.payments"
)

orders_df = (
    spark.table("customer360_dev.silver.orders")
    .select("order_id")
)

customers_df = (
    spark.table("customer360_dev.silver.customers")
    .select("customer_id")
)

In [0]:
typed_df = (
    bronze_df
    .withColumn(
        "payment_attempt_no_cast",
        F.col("payment_attempt_no").cast("int")
    )
    .withColumn(
        "payment_amount_cast",
        F.col("payment_amount").cast("decimal(14,2)")
    )
    .withColumn(
        "refund_amount_cast",
        F.col("refund_amount").cast("decimal(14,2)")
    )
    .withColumn(
        "payment_timestamp_cast",
        F.to_timestamp("payment_timestamp")
    )
    .withColumn(
        "refund_timestamp_cast",
        F.to_timestamp("refund_timestamp")
    )
    .withColumn(
        "created_at_cast",
        F.to_timestamp("created_at")
    )
    .withColumn(
        "updated_at_cast",
        F.to_timestamp("updated_at")
    )
    .withColumn(
        "payment_method_clean",
        F.upper(F.trim("payment_method"))
    )
    .withColumn(
        "payment_status_clean",
        F.upper(F.trim("payment_status"))
    )
)

Add order and customer referential integrity

In [0]:
validated_df = (
    typed_df
    .join(
        orders_df.withColumn(
            "_order_exists",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .join(
        customers_df.withColumn(
            "_customer_exists",
            F.lit(True)
        ),
        on="customer_id",
        how="left"
    )
)

In [0]:
display(validated_df.limit(10))

customer_id,order_id,payment_id,payment_attempt_no,payment_method,payment_status,payment_amount,transaction_reference,failure_reason,payment_timestamp,refund_amount,refund_timestamp,created_at,updated_at,_ingestion_timestamp,_source_file,_source_system,_load_type,payment_attempt_no_cast,payment_amount_cast,refund_amount_cast,payment_timestamp_cast,refund_timestamp_cast,created_at_cast,updated_at_cast,payment_method_clean,payment_status_clean,_order_exists,_customer_exists
CUST000786397,ORD0003752141,PAY00003752141,3,WALLET,REFUNDED,9123.69,TXN000003752141,null,2026-06-15 11:31:53,9123.69,2026-09-23 15:51:33,2026-06-15 11:31:53,2026-06-20 22:26:58,2026-09-26T10:00:20.320Z,dbfs:/Volumes/customer360_dev/staging/source_files/payments/initial/payments_initial_016.csv,PAYMENT_PLATFORM,INITIAL,3,9123.69,9123.69,2026-06-15T11:31:53.000Z,2026-09-23T15:51:33.000Z,2026-06-15T11:31:53.000Z,2026-06-20T22:26:58.000Z,WALLET,REFUNDED,true,true
CUST000759248,ORD0003750544,PAY00003750544,2,NETBANKING,FAILED,7382.96,TXN000003750544,TIMEOUT,2024-10-25 16:44:52,0.00,null,2024-10-25 16:44:52,2024-10-30 03:39:57,2026-09-26T10:00:20.320Z,dbfs:/Volumes/customer360_dev/staging/source_files/payments/initial/payments_initial_016.csv,PAYMENT_PLATFORM,INITIAL,2,7382.96,0.00,2024-10-25T16:44:52.000Z,null,2024-10-25T16:44:52.000Z,2024-10-30T03:39:57.000Z,NETBANKING,FAILED,true,true
CUST000764450,ORD0003750850,PAY00003750850,2,UPI,INITIATED,7716.50,TXN000003750850,null,2025-08-27 22:50:10,0.00,null,2025-08-27 22:50:10,2025-09-01 09:45:15,2026-09-26T10:00:20.320Z,dbfs:/Volumes/customer360_dev/staging/source_files/payments/initial/payments_initial_016.csv,PAYMENT_PLATFORM,INITIAL,2,7716.50,0.00,2025-08-27T22:50:10.000Z,null,2025-08-27T22:50:10.000Z,2025-09-01T09:45:15.000Z,UPI,INITIATED,true,true
CUST000784119,ORD0003752007,PAY00003752007,1,COD,SUCCESS,8977.63,TXN000003752007,null,2026-02-01 09:57:51,0.00,null,2026-02-01 09:57:51,2026-02-06 20:52:56,2026-09-26T10:00:20.320Z,dbfs:/Volumes/customer360_dev/staging/source_files/payments/initial/payments_initial_016.csv,PAYMENT_PLATFORM,INITIAL,1,8977.63,0.00,2026-02-01T09:57:51.000Z,null,2026-02-01T09:57:51.000Z,2026-02-06T20:52:56.000Z,COD,SUCCESS,true,true
CUST000791531,ORD0003752443,PAY00003752443,2,CARD,PARTIALLY_REFUNDED,9452.87,TXN000003752443,null,2024-07-18 13:53:19,3781.15,2024-10-26 17:13:59,2024-07-18 13:53:19,2024-07-23 00:48:24,2026-09-26T10:00:20.320Z,dbfs:/Volumes/customer360_dev/staging/source_files/payments/initial/payments_initial_016.csv,PAYMENT_PLATFORM,INITIAL,2,9452.87,3781.15,2024-07-18T13:53:19.000Z,2024-10-26T17:13:59.000Z,2024-07-18T13:53:19.000Z,2024-07-23T00:48:24.000Z,CARD,PARTIALLY_REFUNDED,true,true
CUST000756426,ORD0003750378,PAY00003750378,1,CARD,PARTIALLY_REFUNDED,7202.02,TXN000003750378,null,2024-05-12 06:18:54,2880.81,2024-08-20 10:38:34,2024-05-12 06:18:54,2024-05-17 17:13:59,2026-09-26T10:00:20.320Z,dbfs:/Volumes/customer360_dev/staging/source_files/payments/initial/payments_initial_016.csv,PAYMENT_PLATFORM,INITIAL,1,7202.02,2880.81,2024-05-12T06:18:54.000Z,2024-08-20T10:38:34.000Z,2024-05-12T06:18:54.000Z,2024-05-17T17:13:59.000Z,CARD,PARTIALLY_REFUNDED,true,true
CUST000761866,ORD0003750698,PAY00003750698,3,CARD,PARTIALLY_REFUNDED,7550.82,TXN000003750698,null,2025-03-28 14:58:14,3020.33,2025-07-06 18:18:54,2025-03-28 14:58:14,2025-04-02 01:53:19,2026-09-26T10:00:20.320Z,dbfs:/Volumes/customer360_dev/staging/source_files/payments/initial/payments_initial_016.csv,PAYMENT_PLATFORM,INITIAL,3,7550.82,3020.33,2025-03-28T14:58:14.000Z,2025-07-06T18:18:54.000Z,2025-03-28T14:58:14.000Z,2025-04-02T01:53:19.000Z,CARD,PARTIALLY_REFUNDED,true,true
CUST000764297,ORD0003750841,PAY00003750841,2,WALLET,REFUNDED,7706.69,TXN000003750841,null,2025-08-18 07:11:13,7706.69,2025-11-26 11:31:53,2025-08-18 07:11:13,2025-08-23 18:06:18,2026-09-26T10:00:20.320Z,dbfs:/Volumes/customer360_dev/staging/source_files/payments/initial/payments_initial_016.csv,PAYMENT_PLATFORM,INITIAL,2,7706.69,7706.69,2025-08-1

In [0]:
validated_df = (
    validated_df
    .withColumn(
        "dq_missing_payment_id",
        F.col("payment_id").isNull()
        | (F.trim(F.col("payment_id")) == "")
    )
    .withColumn(
        "dq_missing_order_id",
        F.col("order_id").isNull()
        | (F.trim(F.col("order_id")) == "")
    )
    .withColumn(
        "dq_missing_customer_id",
        F.col("customer_id").isNull()
        | (F.trim(F.col("customer_id")) == "")
    )
    .withColumn(
        "dq_order_not_found",
        F.col("_order_exists").isNull()
    )
    .withColumn(
        "dq_customer_not_found",
        F.col("_customer_exists").isNull()
    )
    .withColumn(
        "dq_invalid_attempt_no",
        F.col("payment_attempt_no_cast").isNull()
        | (F.col("payment_attempt_no_cast") <= 0)
    )
    .withColumn(
        "dq_invalid_payment_amount",
        F.col("payment_amount_cast").isNull()
        | (F.col("payment_amount_cast") < 0)
    )
    .withColumn(
        "dq_invalid_refund_amount",
        F.col("refund_amount_cast").isNotNull()
        & (F.col("refund_amount_cast") < 0)
    )
    .withColumn(
        "dq_invalid_payment_timestamp",
        F.col("payment_timestamp_cast").isNull()
    )
)

In [0]:
validated_df = validated_df.withColumn(
    "dq_invalid_status",
    ~F.col("payment_status_clean").isin(
        "INITIATED",
        "SUCCESS",
        "FAILED",
        "REFUNDED",
        "PARTIALLY_REFUNDED"
    )
)

In [0]:
validated_df = (
    validated_df
    .withColumn(
        "dq_refund_exceeds_payment",
        F.col("refund_amount_cast").isNotNull()
        & (
            F.col("refund_amount_cast")
            > F.col("payment_amount_cast")
        )
    )
)

In [0]:
validated_df = validated_df.withColumn(
    "dq_warning_refund_status_without_amount",
    (
        F.col("payment_status_clean").isin(
            "REFUNDED",
            "PARTIALLY_REFUNDED"
        )
        & F.col("refund_amount_cast").isNull()
    )
)

In [0]:
validated_df = validated_df.withColumn(
    "dq_warning_refund_without_timestamp",
    (
        F.col("refund_amount_cast").isNotNull()
        & (F.col("refund_amount_cast") > 0)
        & F.col("refund_timestamp_cast").isNull()
    )
)

Detect suspicious duplicate transaction references

In [0]:
successful_txn_duplicates_df = (
    validated_df
    .filter(
        (F.col("payment_status_clean") == "SUCCESS")
        & F.col("transaction_reference").isNotNull()
    )
    .groupBy("transaction_reference")
    .count()
    .filter(F.col("count") > 1)
    .select("transaction_reference")
    .withColumn(
        "_duplicate_success_txn_ref",
        F.lit(True)
    )
)

validated_df = (
    validated_df
    .join(
        successful_txn_duplicates_df,
        on="transaction_reference",
        how="left"
    )
    .withColumn(
        "dq_warning_duplicate_success_txn_ref",
        F.coalesce(
            F.col("_duplicate_success_txn_ref"),
            F.lit(False)
        )
    )
)

In [0]:
validated_df = validated_df.withColumn(
    "dq_critical_failure",
    F.col("dq_missing_payment_id")
    | F.col("dq_missing_order_id")
    | F.col("dq_missing_customer_id")
    | F.col("dq_order_not_found")
    | F.col("dq_customer_not_found")
    | F.col("dq_invalid_attempt_no")
    | F.col("dq_invalid_payment_amount")
    | F.col("dq_invalid_refund_amount")
    | F.col("dq_invalid_payment_timestamp")
    | F.col("dq_invalid_status")
    | F.col("dq_refund_exceeds_payment")
)

In [0]:
display(validated_df.limit(10))

transaction_reference,customer_id,order_id,payment_id,payment_attempt_no,payment_method,payment_status,payment_amount,failure_reason,payment_timestamp,refund_amount,refund_timestamp,created_at,updated_at,_ingestion_timestamp,_source_file,_source_system,_load_type,payment_attempt_no_cast,payment_amount_cast,refund_amount_cast,payment_timestamp_cast,refund_timestamp_cast,created_at_cast,updated_at_cast,payment_method_clean,payment_status_clean,_order_exists,_customer_exists,dq_missing_payment_id,dq_missing_order_id,dq_missing_customer_id,dq_order_not_found,dq_customer_not_found,dq_invalid_attempt_no,dq_invalid_payment_amount,dq_invalid_refund_amount,dq_invalid_payment_timestamp,dq_invalid_status,dq_refund_exceeds_payment,dq_warning_refund_status_without_amount,dq_warning_refund_without_timestamp,_duplicate_success_txn_ref,dq_warning_duplicate_success_txn_ref,dq_critical_failure
TXN000003750733,CUST000762461,ORD0003750733,PAY00003750733,2,CARD,PARTIALLY_REFUNDED,7588.97,null,2025-05-02 19:23:49,3035.59,2025-08-10 23:43:29,2025-05-02 19:23:49,2025-05-07 06:18:54,2026-09-26T10:00:20.320Z,dbfs:/Volumes/customer360_dev/staging/source_files/payments/initial/payments_initial_016.csv,PAYMENT_PLATFORM,INITIAL,2,7588.97,3035.59,2025-05-02T19:23:49.000Z,2025-08-10T23:43:29.000Z,2025-05-02T19:23:49.000Z,2025-05-07T06:18:54.000Z,CARD,PARTIALLY_REFUNDED,true,true,false,false,false,false,false,false,false,false,false,false,false,false,false,null,false,false
TXN000003751081,CUST000768377,ORD0003751081,PAY00003751081,2,WALLET,REFUNDED,7968.29,null,2026-04-15 07:11:13,7968.29,2026-07-24 11:31:53,2026-04-15 07:11:13,2026-04-20 18:06:18,2026-09-26T10:00:20.320Z,dbfs:/Volumes/customer360_dev/staging/source_files/payments/initial/payments_initial_016.csv,PAYMENT_PLATFORM,INITIAL,2,7968.29,7968.29,2026-04-15T07:11:13.000Z,2026-07-24T11:31:53.000Z,2026-04-15T07:11:13.000Z,2026-04-20T18:06:18.000Z,WALLET,REFUNDED,true,true,false,false,false,false,false,false,false,false,false,false,false,false,false,null,false,false
TXN000003753164,CUST000803788,ORD0003753164,PAY00003753164,3,NETBANKING,FAILED,10238.76,BANK_DECLINED,2026-07-09 20:04:32,0.00,null,2026-07-09 20:04:32,2026-07-14 07:59:37,2026-09-26T10:00:20.320Z,dbfs:/Volumes/customer360_dev/staging/source_files/payments/initial/payments_initial_016.csv,PAYMENT_PLATFORM,INITIAL,3,10238.76,0.00,2026-07-09T20:04:32.000Z,null,2026-07-09T20:04:32.000Z,2026-07-14T07:59:37.000Z,NETBANKING,FAILED,true,true,false,false,false,false,false,false,false,false,false,false,false,false,false,null,false,false
TXN000003750795,CUST000763515,ORD0003750795,PAY00003750795,1,UPI,INITIATED,7656.55,null,2025-07-03 21:45:15,0.00,null,2025-07-03 21:45:15,2025-07-08 08:40:20,2026-09-26T10:00:20.320Z,dbfs:/Volumes/customer360_dev/staging/source_files/payments/initial/payments_initial_016.csv,PAYMENT_PLATFORM,INITIAL,1,7656.55,0.00,2025-07-03T21:45:15.000Z,null,2025-07-03T21:45:15.000Z,2025-07-08T08:40:20.000Z,UPI,INITIATED,true,true,false,false,false,false,false,false,false,false,false,false,false,false,false,null,false,false
TXN000003753566,CUST000810622,ORD0003753566,PAY00003753566,3,WALLET,REFUNDED,10676.94,null,2024-11-19 02:46:38,10676.94,2025-02-27 06:06:18,2024-11-19 02:46:38,2024-11-24 13:41:43,2026-09-26T10:00:20.320Z,dbfs:/Volumes/customer360_dev/staging/source_files/payments/initial/payments_initial_016.csv,PAYMENT_PLATFORM,INITIAL,3,10676.94,10676.94,2024-11-19T02:46:38.000Z,2025-02-27T06:06:18.000Z,2024-11-19T02:46:38.000Z,2024-11-24T13:41:43.000Z,WALLET,REFUNDED,true,true,false,false,false,false,false,false,false,false,false,false,false,false,false,null,false,false
TXN000003754230,CUST000821910,ORD0003754230,PAY00003754230,1,UPI,INITIATED,11400.70,null,2026-09-14 18:30:30,0.00,null,2026-09-14 18:30:30,2026-09-19 05:25:35,2026-09-26T10:00:20.320Z,dbfs:/Volumes/customer360_dev/staging/source_files/payments/initial/payments_initial_016.csv,PAYMENT_PLATFORM,INITIAL,1,11400.70,0.00,2026-09-14T18:30:30.000Z,null,2026-09-14T18:

In [0]:
reject_df = validated_df.filter(
    F.col("dq_critical_failure")
)

valid_df = validated_df.filter(
    ~F.col("dq_critical_failure")
)

In [0]:
payment_window = (
    Window
    .partitionBy("payment_id")
    .orderBy(
        F.col("updated_at_cast").desc_nulls_last(),
        F.col("_ingestion_timestamp").desc()
    )
)

deduped_df = (
    valid_df
    .withColumn(
        "_row_number",
        F.row_number().over(payment_window)
    )
    .filter(F.col("_row_number") == 1)
)

In [0]:
silver_df = deduped_df.select(
    "payment_id",
    "order_id",
    "customer_id",

    F.col("payment_attempt_no_cast").alias(
        "payment_attempt_no"
    ),

    F.col("payment_method_clean").alias(
        "payment_method"
    ),

    F.col("payment_status_clean").alias(
        "payment_status"
    ),

    F.col("payment_amount_cast").alias(
        "payment_amount"
    ),

    "transaction_reference",
    "failure_reason",

    F.col("payment_timestamp_cast").alias(
        "payment_timestamp"
    ),

    F.col("refund_amount_cast").alias(
        "refund_amount"
    ),

    F.col("refund_timestamp_cast").alias(
        "refund_timestamp"
    ),

    "dq_warning_refund_status_without_amount",
    "dq_warning_refund_without_timestamp",
    "dq_warning_duplicate_success_txn_ref",

    F.col("created_at_cast").alias("created_at"),
    F.col("updated_at_cast").alias("updated_at"),

    "_source_file",
    "_source_system",
    "_load_type",
    "_ingestion_timestamp",

    F.current_timestamp().alias(
        "_silver_processed_at"
    )
)

In [0]:
(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "customer360_dev.silver.payments"
    )
)

In [0]:
quarantine_df = (
    reject_df
    .withColumn(
        "rejection_reason",
        F.concat_ws(
            "; ",
            F.when(
                F.col("dq_missing_payment_id"),
                F.lit("MISSING_PAYMENT_ID")
            ),
            F.when(
                F.col("dq_missing_order_id"),
                F.lit("MISSING_ORDER_ID")
            ),
            F.when(
                F.col("dq_missing_customer_id"),
                F.lit("MISSING_CUSTOMER_ID")
            ),
            F.when(
                F.col("dq_order_not_found"),
                F.lit("ORDER_NOT_FOUND")
            ),
            F.when(
                F.col("dq_customer_not_found"),
                F.lit("CUSTOMER_NOT_FOUND")
            ),
            F.when(
                F.col("dq_invalid_attempt_no"),
                F.lit("INVALID_PAYMENT_ATTEMPT_NO")
            ),
            F.when(
                F.col("dq_invalid_payment_amount"),
                F.lit("INVALID_PAYMENT_AMOUNT")
            ),
            F.when(
                F.col("dq_invalid_refund_amount"),
                F.lit("INVALID_REFUND_AMOUNT")
            ),
            F.when(
                F.col("dq_invalid_payment_timestamp"),
                F.lit("INVALID_PAYMENT_TIMESTAMP")
            ),
            F.when(
                F.col("dq_invalid_status"),
                F.lit("INVALID_PAYMENT_STATUS")
            ),
            F.when(
                F.col("dq_refund_exceeds_payment"),
                F.lit("REFUND_EXCEEDS_PAYMENT")
            )
        )
    )
    .withColumn(
        "_quarantined_at",
        F.current_timestamp()
    )
)

In [0]:
if "_pipeline_run_id" not in spark.table("customer360_dev.quarantine.payment_rejects").columns:
    spark.sql("ALTER TABLE customer360_dev.quarantine.payment_rejects ADD COLUMNS (_pipeline_run_id STRING)")
(
    quarantine_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "customer360_dev.quarantine.payment_rejects"
    )
)

In [0]:
bronze_count = spark.table(
    "customer360_dev.bronze.payments"
).count()

silver_count = spark.table(
    "customer360_dev.silver.payments"
).count()

reject_count = spark.table(
    "customer360_dev.quarantine.payment_rejects"
).count()

print(f"Bronze Payments : {bronze_count:,}")
print(f"Silver Payments : {silver_count:,}")
print(f"Rejected        : {reject_count:,}")

Bronze Payments : 6,000,150
Silver Payments : 5,652,000
Rejected        : 348,150
